# MIMIC-IV Data Cleaning & Preprocessing Pipeline
**MRes Computing and Artificial Intelligence — MRES7015**
Federated Learning for Privacy-Preserving EHR Analysis in the UK NHS
University of Greater Manchester

---

## Correct Path Configuration
| Item | Path |
|---|---|
| ZIP source | `/content/drive/MyDrive/PROJECT DATASET/mimic-iv-3.zip` |
| Extraction target | `/tmp/mimic-iv-data/` |
| Data subfolder | `mimic-iv-3.1` |
| Artefacts (permanent) | `/content/drive/MyDrive/FL_Dissertation/data/artefacts/` |
| Cleaned outputs | `/content/drive/MyDrive/FL_Dissertation/data/cleaned/` |

## Pipeline Stages
1. Environment setup and path configuration
2. Mount Drive and extract ZIP
3. Discover and load MIMIC-IV tables
4. Merge core tables into one analytical DataFrame
5. Duplicate detection and removal
6. Column-level quality assessment
7. Missing value analysis and treatment
8. Outlier detection and winsorisation
9. Data type correction
10. Feature engineering (NHS-aligned)
11. Categorical encoding
12. Feature selection
13. Class imbalance assessment
14. Normalisation, train/test split, and save

> Run cells **top to bottom**. Each stage prints diagnostic output.
> All artefacts are saved to Google Drive so they survive session disconnects.


## Stage 1 — Environment Setup and Path Configuration

In [ ]:
# Install required libraries
!pip install scikit-learn imbalanced-learn scipy psutil -q
print("Libraries ready")

Libraries ready


In [ ]:
import os, json, pickle, gc, warnings, psutil, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── PATH CONFIGURATION ─────────────────────────────────────────────────────
ZIP_PATH       = '/content/drive/MyDrive/PROJECT DATASET/mimic-iv-3.zip'
EXTRACT_PATH   = '/tmp/mimic-iv-data/'
DATA_SUBFOLDER = 'mimic-iv-3.1'
ACTUAL_DATA    = os.path.join(EXTRACT_PATH, DATA_SUBFOLDER)

# Permanent storage back to Drive (survives session disconnects)
DRIVE_BASE     = '/content/drive/MyDrive/FL_Dissertation/'
ARTEFACTS      = os.path.join(DRIVE_BASE, 'data/artefacts/')
CLEAN_DIR      = os.path.join(DRIVE_BASE, 'data/cleaned/')
PLOT_DIR       = os.path.join(DRIVE_BASE, 'data/diagnostics/')

# Configuration
OUTCOME_COL  = 'hospital_expire_flag'   # binary: 1 = died in hospital
SUBJECT_COL  = 'subject_id'
HADM_COL     = 'hadm_id'
RANDOM_SEED  = 42

def get_memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

print("Path configuration set")
print(f"ZIP source:    {ZIP_PATH}")
print(f"Extract to:    {EXTRACT_PATH}")
print(f"Artefacts:     {ARTEFACTS}")
print(f"Memory usage:  {get_memory_mb():.1f} MB")

Path configuration set
ZIP source:    /content/drive/MyDrive/PROJECT DATASET/mimic-iv-3.zip
Extract to:    /tmp/mimic-iv-data/
Artefacts:     /content/drive/MyDrive/FL_Dissertation/data/artefacts/
Memory usage:  247.2 MB


## Stage 2 — Mount Google Drive and Extract ZIP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

Mounted at /content/drive
Google Drive mounted


In [ ]:
# Create Drive output directories
for d in [ARTEFACTS, CLEAN_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  Ready: {d}")

# Extract ZIP to /tmp (fast local SSD — much quicker than reading from Drive)
if not os.path.exists(ZIP_PATH):
    print(f"ERROR: ZIP not found at {ZIP_PATH}")
    print("Check your Google Drive folder name — it must match exactly including spaces")
else:
    print(f"ZIP found: {ZIP_PATH}")

    # Clear previous extraction if it exists
    if os.path.exists(EXTRACT_PATH):
        print(f"Clearing previous extraction at {EXTRACT_PATH}")
        shutil.rmtree(EXTRACT_PATH)
    os.makedirs(EXTRACT_PATH, exist_ok=True)

    print("Extracting ZIP — this may take 5-10 minutes for MIMIC-IV...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_PATH)
    print("Extraction complete")
    print(f"Memory usage: {get_memory_mb():.1f} MB")

  Ready: /content/drive/MyDrive/FL_Dissertation/data/artefacts/
  Ready: /content/drive/MyDrive/FL_Dissertation/data/cleaned/
  Ready: /content/drive/MyDrive/FL_Dissertation/data/diagnostics/
ZIP found: /content/drive/MyDrive/PROJECT DATASET/mimic-iv-3.zip
Extracting ZIP — this may take 5-10 minutes for MIMIC-IV...
Extraction complete
Memory usage: 247.3 MB


In [ ]:
# Verify extraction and list available tables
import subprocess

print(f"Contents of {EXTRACT_PATH}:")
result = subprocess.run(['ls', '-la', EXTRACT_PATH], capture_output=True, text=True)
print(result.stdout)

if os.path.exists(ACTUAL_DATA):
    print(f"\nContents of {ACTUAL_DATA}:")
    for item in sorted(os.listdir(ACTUAL_DATA)):
        full = os.path.join(ACTUAL_DATA, item)
        if os.path.isdir(full):
            print(f"  [{item}/]")
            for f in sorted(os.listdir(full)):
                size = os.path.getsize(os.path.join(full,f)) / (1024**2)
                print(f"    {f}  ({size:.1f} MB)")
else:
    print(f"WARNING: Expected subfolder not found: {ACTUAL_DATA}")
    print("Check DATA_SUBFOLDER variable — your ZIP may use a different folder name")
    print("Actual contents:", os.listdir(EXTRACT_PATH))

Contents of /tmp/mimic-iv-data/:
total 12
drwxr-xr-x 3 root root 4096 Jul  7 12:21 .
drwxrwxrwt 1 root root 4096 Jul  7 12:21 ..
drwxr-xr-x 4 root root 4096 Jul  7 12:24 mimic-iv-3.1


Contents of /tmp/mimic-iv-data/mimic-iv-3.1:
  [hosp/]
    admissions.csv.gz  (19.0 MB)
    d_hcpcs.csv.gz  (0.4 MB)
    d_icd_diagnoses.csv.gz  (0.8 MB)
    d_icd_procedures.csv.gz  (0.6 MB)
    d_labitems.csv.gz  (0.0 MB)
    diagnoses_icd.csv.gz  (32.0 MB)
    drgcodes.csv.gz  (9.3 MB)
    emar.csv.gz  (773.7 MB)
    emar_detail.csv.gz  (713.5 MB)
    hcpcsevents.csv.gz  (2.1 MB)
    labevents.csv.gz  (2472.8 MB)
    microbiologyevents.csv.gz  (112.2 MB)
    omr.csv.gz  (42.0 MB)
    patients.csv.gz  (2.7 MB)
    pharmacy.csv.gz  (501.4 MB)
    poe.csv.gz  (635.7 MB)
    poe_detail.csv.gz  (52.7 MB)
    prescriptions.csv.gz  (578.2 MB)
    procedures_icd.csv.gz  (7.4 MB)
    provider.csv.gz  (0.1 MB)
    services.csv.gz  (8.2 MB)
    transfers.csv.gz  (44.0 MB)
  [icu/]
    caregiver.csv.gz  (0.0 MB)


## Stage 3 — Discover and Load MIMIC-IV Tables

MIMIC-IV is organised into two modules:
- **hosp/** — hospital-level data: admissions, patients, diagnoses, lab events
- **icu/** — ICU-level data: icustays, chartevents, inputevents

We load the tables needed for in-hospital mortality prediction and store paths
for large tables (chartevents, labevents) to load on demand.


In [ ]:
mimic_dfs = {}           # holds DataFrames loaded in memory
mimic_file_paths = {}    # holds paths for large files (load on demand)

# Tables to load directly into memory (smaller, always needed)
LOAD_DIRECT = [
    'admissions',     # core admission info + outcome
    'patients',       # demographics
    'icustays',       # ICU stay details
    'diagnoses_icd',  # ICD-10 diagnosis codes
]

# Tables to index by path only (large — load only if needed)
LARGE_TABLES = ['chartevents', 'labevents', 'inputevents', 'outputevents']

print(f"Scanning {ACTUAL_DATA} for CSV files...")

for root, dirs, files in os.walk(ACTUAL_DATA):
    for fname in sorted(files):
        if not fname.endswith('.csv.gz'):
            continue

        table_name = fname.replace('.csv.gz', '')
        full_path  = os.path.join(root, fname)
        size_mb    = os.path.getsize(full_path) / (1024**2)

        mimic_file_paths[table_name] = full_path

        if table_name in LOAD_DIRECT:
            print(f"  Loading: {table_name} ({size_mb:.1f} MB)...")
            try:
                df = pd.read_csv(full_path, compression='gzip', low_memory=False)
                mimic_dfs[table_name] = df
                print(f"    Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
            except Exception as e:
                print(f"    ERROR loading {table_name}: {e}")
        else:
            print(f"  Indexed: {table_name} ({size_mb:.1f} MB) — path stored")

        gc.collect()

print(f"\nTables in memory: {list(mimic_dfs.keys())}")
print(f"Tables indexed:   {list(mimic_file_paths.keys())}")
print(f"Memory usage:     {get_memory_mb():.1f} MB")

Scanning /tmp/mimic-iv-data/mimic-iv-3.1 for CSV files...
  Loading: admissions (19.0 MB)...
    Loaded: 546,028 rows x 16 cols
  Indexed: d_hcpcs (0.4 MB) — path stored
  Indexed: d_icd_diagnoses (0.8 MB) — path stored
  Indexed: d_icd_procedures (0.6 MB) — path stored
  Indexed: d_labitems (0.0 MB) — path stored
  Loading: diagnoses_icd (32.0 MB)...
    Loaded: 6,364,488 rows x 5 cols
  Indexed: drgcodes (9.3 MB) — path stored
  Indexed: emar (773.7 MB) — path stored
  Indexed: emar_detail (713.5 MB) — path stored
  Indexed: hcpcsevents (2.1 MB) — path stored
  Indexed: labevents (2472.8 MB) — path stored
  Indexed: microbiologyevents (112.2 MB) — path stored
  Indexed: omr (42.0 MB) — path stored
  Loading: patients (2.7 MB)...
    Loaded: 364,627 rows x 6 cols
  Indexed: pharmacy (501.4 MB) — path stored
  Indexed: poe (635.7 MB) — path stored
  Indexed: poe_detail (52.7 MB) — path stored
  Indexed: prescriptions (578.2 MB) — path stored
  Indexed: procedures_icd (7.4 MB) — path st

In [ ]:
# Quick inspection of core tables
for name in ['admissions', 'patients', 'icustays']:
    if name in mimic_dfs:
        df = mimic_dfs[name]
        print(f"\n{'='*55}")
        print(f"TABLE: {name.upper()}  |  {df.shape[0]:,} rows x {df.shape[1]} cols")
        print('='*55)
        print("Columns:", df.columns.tolist())
        display(df.head(3))


TABLE: ADMISSIONS  |  546,028 rows x 16 cols
Columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0



TABLE: PATIENTS  |  364,627 rows x 6 cols
Columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN



TABLE: ICUSTAYS  |  94,458 rows x 8 cols
Columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.4103
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.8933
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.4975


## Stage 4 — Merge Core Tables into One Analytical DataFrame

Join admissions + patients + icustays on subject_id/hadm_id to create a single
row-per-admission analytical dataset for binary mortality prediction.


In [ ]:
# Validate required tables are loaded
required = ['admissions', 'patients', 'icustays']
missing_tables = [t for t in required if t not in mimic_dfs]
if missing_tables:
    raise RuntimeError(f"Missing required tables: {missing_tables}. Check Stage 3.")

admissions = mimic_dfs['admissions'].copy()
patients   = mimic_dfs['patients'].copy()
icustays   = mimic_dfs['icustays'].copy()

print(f"Admissions: {admissions.shape}")
print(f"Patients:   {patients.shape}")
print(f"ICUstays:   {icustays.shape}")

# Check outcome column exists
if OUTCOME_COL not in admissions.columns:
    print(f"\nWARNING: '{OUTCOME_COL}' not in admissions columns")
    print("Columns that might be the outcome:")
    print([c for c in admissions.columns if any(
        k in c.lower() for k in ['expire','death','mort','discharge']
    )])
else:
    vc = admissions[OUTCOME_COL].value_counts()
    print(f"\nOutcome '{OUTCOME_COL}':")
    print(f"  Survived (0): {vc.get(0,0):>8,}")
    print(f"  Died     (1): {vc.get(1,0):>8,}")
    print(f"  Positive rate: {admissions[OUTCOME_COL].mean():.3f}")

Admissions: (546028, 16)
Patients:   (364627, 6)
ICUstays:   (94458, 8)

Outcome 'hospital_expire_flag':
  Survived (0):  534,227
  Died     (1):   11,801
  Positive rate: 0.022


In [ ]:
# Step 4a — Merge admissions with patients
df = admissions.merge(
    patients[['subject_id','gender','anchor_age','anchor_year']],
    on='subject_id', how='left'
)
print(f"After admissions + patients merge: {df.shape}")

# Step 4b — Merge with ICU stays (keep first ICU stay per admission)
icu_first = (icustays
    .sort_values('intime')
    .groupby(['subject_id','hadm_id'])
    .first()
    .reset_index()
    [['subject_id','hadm_id','stay_id','los','first_careunit','last_careunit']]
)
icu_first.columns = ['subject_id','hadm_id','stay_id','icu_los',
                     'first_careunit','last_careunit']

df = df.merge(icu_first, on=['subject_id','hadm_id'], how='left')
print(f"After + ICU stays merge:          {df.shape}")

# Step 4c — Compute derived time features from admission timestamps
date_cols = ['admittime','dischtime','edregtime','edouttime']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'admittime' in df.columns and 'dischtime' in df.columns:
    df['los_hospital_hours'] = (
        df['dischtime'] - df['admittime']
    ).dt.total_seconds() / 3600
    df['los_hospital_hours'] = df['los_hospital_hours'].clip(lower=0)
    print(f"Hospital LOS computed (hours): mean={df['los_hospital_hours'].mean():.1f}")

# Drop raw timestamp columns (not useful as features)
df = df.drop(columns=[c for c in date_cols if c in df.columns], errors='ignore')

# Use anchor_age as patient age
if 'anchor_age' in df.columns:
    df = df.rename(columns={'anchor_age': 'age'})

print(f"\nFinal merged shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Store as working DataFrame
mimic_dfs['merged'] = df
print("\nMerged DataFrame stored as mimic_dfs['merged']")

After admissions + patients merge: (546028, 19)
After + ICU stays merge:          (546028, 23)
Hospital LOS computed (hours): mean=114.3

Final merged shape: (546028, 20)
Columns: ['subject_id', 'hadm_id', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'hospital_expire_flag', 'gender', 'age', 'anchor_year', 'stay_id', 'icu_los', 'first_careunit', 'last_careunit', 'los_hospital_hours']

Merged DataFrame stored as mimic_dfs['merged']


## Stage 5 — Duplicate Detection and Removal

## Stage 4b — Leakage Prevention

Removes columns recorded **at or after discharge** that directly encode mortality. `discharge_location` contains values like DIED and HOSPICE — retaining it causes the model to learn the answer rather than genuine clinical predictors, artificially inflating AUC.

**Must run before Stage 5 (duplicates) and before feature selection.**

In [ ]:
# ── LEAKAGE PREVENTION ─────────────────────────────────────────────────────
# These columns are recorded at or after discharge and directly encode
# the mortality outcome. Keeping them allows the model to learn the answer
# rather than genuine admission-time clinical predictors.
#
# Key offender: discharge_location contains DIED, HOSPICE-HOME etc.
# which were the #1 most informative feature in the initial run (MI=0.1476),
# inflating the centralised baseline to AUC=0.9933.

LEAKAGE_COLUMNS = [
    'discharge_location',   # DIED / HOSPICE — direct outcome encoding
    'deathtime',            # death timestamp — direct leakage
    'edouttime',            # ED discharge time — recorded post-outcome
    'discharge_location_EXPIRED',  # one-hot variant if already encoded
]

# Drop leakage columns — protect outcome column
to_drop = [
    c for c in LEAKAGE_COLUMNS
    if c in df.columns and c != OUTCOME_COL
]

# Also drop any one-hot columns that derived from discharge_location
discharge_ohe = [c for c in df.columns
                 if c.startswith('discharge_location_')]
to_drop += discharge_ohe
to_drop = list(set(to_drop))  # deduplicate

removed = []
for col in to_drop:
    df = df.drop(columns=[col], errors='ignore')
    removed.append(col)

print(f'Leakage columns removed ({len(removed)}):')
for c in removed:
    print(f'  - {c}')
print(f'\nColumns remaining: {df.shape[1]}')
print('Centralised baseline AUC will now reflect genuine clinical '
      'predictors available at admission time.')
mimic_dfs['merged'] = df


Leakage columns removed (2):
  - deathtime
  - discharge_location

Columns remaining: 18
Centralised baseline AUC will now reflect genuine clinical predictors available at admission time.


In [ ]:
df = mimic_dfs['merged'].copy()
initial_rows = len(df)

# 5a — Exact row duplicates
exact_dupes = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Exact duplicate rows removed:     {exact_dupes:>6,}")

# 5b — Duplicate hospital admissions (subject_id + hadm_id)
hadm_dupes = df.duplicated(subset=[SUBJECT_COL, HADM_COL]).sum()
df = df.drop_duplicates(subset=[SUBJECT_COL, HADM_COL], keep='first')
print(f"Duplicate admissions removed:     {hadm_dupes:>6,}")

# 5c — Report multiple admissions per patient (keep all — each is independent)
adm_per_patient = df[SUBJECT_COL].value_counts()
multi = (adm_per_patient > 1).sum()
print(f"\nPatients with multiple admissions: {multi:>5,}")
print(f"Max admissions per patient:        {adm_per_patient.max():>5,}")
print(f"(Multiple admissions kept — each is an independent clinical episode)")

removed = initial_rows - len(df)
print(f"\nRows after deduplication: {len(df):,}  ({removed:,} removed)")
mimic_dfs['merged'] = df

Exact duplicate rows removed:          0
Duplicate admissions removed:          0

Patients with multiple admissions: 100,163
Max admissions per patient:          238
(Multiple admissions kept — each is an independent clinical episode)

Rows after deduplication: 546,028  (0 removed)


## Stage 6 — Column-Level Quality Assessment

In [ ]:
df = mimic_dfs['merged'].copy()
MISSING_THRESHOLD = 0.40
protected = [OUTCOME_COL, SUBJECT_COL, HADM_COL, 'stay_id']

quality_report = pd.DataFrame({
    'dtype':       df.dtypes,
    'n_missing':   df.isnull().sum(),
    'pct_missing': df.isnull().mean().round(4),
    'n_unique':    df.nunique(),
    'pct_unique':  (df.nunique() / len(df)).round(4),
}).reset_index().rename(columns={'index':'column'})

quality_report['status'] = 'keep'
quality_report.loc[
    quality_report['pct_missing'] > MISSING_THRESHOLD, 'status'
] = 'DROP: >40% missing'
quality_report.loc[
    quality_report['n_unique'] <= 1, 'status'
] = 'DROP: zero variance'

# Protect key columns
for col in protected:
    quality_report.loc[quality_report['column'] == col, 'status'] = 'keep (protected)'

drop_cols = quality_report[
    quality_report['status'].str.startswith('DROP')
]['column'].tolist()

print(f"Total columns:     {len(quality_report)}")
print(f"Columns to keep:   {(quality_report['status'].str.startswith('keep')).sum()}")
print(f"Columns to drop:   {len(drop_cols)}")
print(f"\nDropping: {drop_cols}")

df = df.drop(columns=drop_cols, errors='ignore')
print(f"\nColumns remaining: {df.shape[1]}")

quality_report.to_csv(ARTEFACTS + 'column_quality_report.csv', index=False)
print("Quality report saved to Drive")
mimic_dfs['merged'] = df

Total columns:     18
Columns to keep:   15
Columns to drop:   3

Dropping: ['icu_los', 'first_careunit', 'last_careunit']

Columns remaining: 15
Quality report saved to Drive


## Stage 7 — Missing Value Analysis and Treatment

In [ ]:
df = mimic_dfs['merged'].copy()

num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in protected]
cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns
            if c not in protected]

# 7a — Missing indicator flags for columns with >5% missing
INDICATOR_THRESHOLD = 0.05
indicators_added = []
for col in num_cols:
    if df[col].isnull().mean() > INDICATOR_THRESHOLD:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
        indicators_added.append(col)

print(f"Missing indicator columns added: {len(indicators_added)}")

# 7b — Median imputation for numeric columns
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])
print(f"Numeric imputation (median): {len(num_cols)} columns")

# 7c — Mode imputation for categorical columns
if cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
    print(f"Categorical imputation (mode): {len(cat_cols)} columns")

# Verify
remaining = df.drop(columns=protected, errors='ignore').isnull().sum().sum()
print(f"\nRemaining missing values (excl. protected): {remaining}")

# Save imputers
with open(ARTEFACTS + 'num_imputer.pkl', 'wb') as f: pickle.dump(num_imputer, f)
print("Imputers saved to Drive")
mimic_dfs['merged'] = df

# Plot missingness
fig, ax = plt.subplots(figsize=(10,4))
miss = df.isnull().mean().sort_values(ascending=False).head(20)
ax.barh(miss.index, miss.values*100, color='#444')
ax.axvline(40, color='red', linewidth=0.8, linestyle='--', label='Drop threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Top 20 Remaining Columns by Missingness')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR + 'missingness.png', dpi=150)
plt.close()
print("Missingness chart saved")

Missing indicator columns added: 0
Numeric imputation (median): 3 columns
Categorical imputation (mode): 8 columns

Remaining missing values (excl. protected): 0
Imputers saved to Drive
Missingness chart saved


## Stage 8 — Outlier Detection and Winsorisation

In [ ]:
df = mimic_dfs['merged'].copy()

# 8a — Clinical range checks (physiologically plausible bounds for ICU patients)
CLINICAL_RANGES = {
    'heart_rate':        (20,  300),
    'resp_rate':         (4,   60),
    'sbp':               (40,  300),   # systolic BP
    'dbp':               (20,  200),   # diastolic BP
    'temperature':       (25,  45),    # Celsius
    'spo2':              (50,  100),   # SpO2 %
    'glucose':           (1,   50),    # mmol/L
    'age':               (0,   120),
    'icu_los':           (0,   180),   # ICU LOS days
    'los_hospital_hours':(0,   8760),  # max 1 year
}

outlier_log = {}
print("Clinical range clamping:")
for col, (lo, hi) in CLINICAL_RANGES.items():
    if col in df.columns:
        n = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lower=lo, upper=hi)
        outlier_log[col] = int(n)
        if n > 0:
            print(f"  {col:<25} {n:>6,} values clamped to [{lo}, {hi}]")

# 8b — IQR winsorisation for remaining numeric columns
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in protected and c not in CLINICAL_RANGES]
IQR_MULT = 3.0
iqr_log = {}

for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - IQR_MULT*IQR, Q3 + IQR_MULT*IQR
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    if n > 0:
        df[col] = df[col].clip(lower=lo, upper=hi)
        iqr_log[col] = int(n)

print(f"\nIQR winsorisation applied to {len(iqr_log)} columns")
print(f"Total outlier values capped: {sum(iqr_log.values()):,}")

with open(ARTEFACTS + 'outlier_log.json', 'w') as f:
    json.dump({'clinical':outlier_log,'iqr':iqr_log}, f, indent=2)
print("Outlier log saved to Drive")
mimic_dfs['merged'] = df

Clinical range clamping:
  los_hospital_hours             1 values clamped to [0, 8760]

IQR winsorisation applied to 0 columns
Total outlier values capped: 0
Outlier log saved to Drive


## Stage 9 — Data Type Correction

In [ ]:
df = mimic_dfs['merged'].copy()

# 9a — Convert numeric-looking string columns
for col in df.select_dtypes(include='object').columns:
    if col in protected: continue
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.notna().mean() > 0.80:
        df[col] = converted
        print(f"  Converted to numeric: {col}")

# 9b — Ensure outcome is binary integer
if OUTCOME_COL in df.columns:
    df[OUTCOME_COL] = pd.to_numeric(df[OUTCOME_COL], errors='coerce')
    df[OUTCOME_COL] = df[OUTCOME_COL].fillna(0).astype(int)
    invalid = (~df[OUTCOME_COL].isin([0,1])).sum()
    if invalid > 0:
        df = df[df[OUTCOME_COL].isin([0,1])]
        print(f"Removed {invalid} rows with invalid outcome values")
    print(f"Outcome verified: {df[OUTCOME_COL].value_counts().to_dict()}")

# 9c — Encode gender as binary
if 'gender' in df.columns:
    df['gender_female'] = (df['gender'].str.upper() == 'F').astype(int)
    df = df.drop(columns=['gender'])
    print("Gender encoded as binary (gender_female: 1=F, 0=M)")

# Refresh column type lists
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in protected]
cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns if c not in protected]

print(f"\nNumeric columns:     {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
mimic_dfs['merged'] = df

Outcome verified: {0: 534227, 1: 11801}
Gender encoded as binary (gender_female: 1=F, 0=M)

Numeric columns:     4
Categorical columns: 7


## Stage 10 — Feature Engineering (NHS-Aligned)

Creates clinically meaningful derived features aligned with NHS acute Trust EHR systems,
including NEWS2-compatible variables used across all NHS acute Trusts.


In [ ]:
df = mimic_dfs['merged'].copy()
features_added = []

# Shock Index (HR / SBP) — haemodynamic instability indicator
if all(c in df.columns for c in ['heart_rate', 'sbp']):
    df['shock_index'] = (df['heart_rate'] / df['sbp'].replace(0, np.nan)).clip(0,5)
    features_added.append('shock_index')

# Pulse Pressure (SBP - DBP) — cardiovascular status
if all(c in df.columns for c in ['sbp', 'dbp']):
    df['pulse_pressure'] = df['sbp'] - df['dbp']
    features_added.append('pulse_pressure')

# Oxygen Deficit (100 - SpO2) — respiratory compromise proxy
if 'spo2' in df.columns:
    df['spo2_deficit'] = 100 - df['spo2']
    features_added.append('spo2_deficit')

# NHS NEWS2 Proxy Score
news2_map = {
    'resp_rate':   [(0,11,0),(12,20,0),(21,24,2),(25,999,3)],
    'spo2':        [(93,100,0),(91,92,1),(89,90,2),(0,88,3)],
    'sbp':         [(111,219,0),(101,110,1),(91,100,2),(0,90,3),(220,999,3)],
    'heart_rate':  [(51,90,0),(41,50,1),(91,110,1),(111,130,2),(0,40,3),(131,999,3)],
    'temperature': [(36.1,38.0,0),(35.1,36.0,1),(38.1,39.0,1),(39.1,999,2),(0,35.0,3)],
}

def news2_pts(val, ranges):
    for lo, hi, pts in ranges:
        if lo <= val <= hi: return pts
    return 0

avail_news2 = [c for c in news2_map if c in df.columns]
if len(avail_news2) >= 3:
    score = np.zeros(len(df))
    for comp in avail_news2:
        score += df[comp].apply(lambda v: news2_pts(v, news2_map[comp])).values
    df['news2_proxy_score'] = score
    features_added.append('news2_proxy_score')
    print(f"NEWS2 proxy score computed from: {avail_news2}")

# Age groups (NHS clinical audit standard boundaries)
if 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'],
        bins=[0,18,40,60,75,120],
        labels=['0-18','19-40','41-60','61-75','76+']).astype(str)
    features_added.append('age_group')

# ICU Care Unit category (simplified)
if 'first_careunit' in df.columns:
    icu_map = {'Medical Intensive Care Unit (MICU)':'MICU',
               'Surgical Intensive Care Unit (SICU)':'SICU',
               'Cardiac Vascular Intensive Care Unit (CVICU)':'CVICU',
               'Coronary Care Unit (CCU)':'CCU',
               'Neuro Surgical Intensive Care Unit (Neuro SICU)':'NeuroSICU',
               'Medical/Surgical Intensive Care Unit (MICU/SICU)':'MICU_SICU'}
    df['icu_type'] = df['first_careunit'].map(icu_map).fillna('Other')
    features_added.append('icu_type')

print(f"\nFeature engineering complete")
print(f"New features: {len(features_added)}")
for f in features_added:
    print(f"  + {f}")

mimic_dfs['merged'] = df


Feature engineering complete
New features: 1
  + age_group


## Stage 11 — Categorical Encoding

In [ ]:
df = mimic_dfs['merged'].copy()

cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns
            if c not in protected]

low_card  = [c for c in cat_cols if df[c].nunique() <= 10]
high_card = [c for c in cat_cols if df[c].nunique() >  10]

print(f"Low-cardinality (one-hot):  {low_card}")
print(f"High-cardinality (ordinal): {high_card}")

# One-hot encode low-cardinality columns
if low_card:
    df = pd.get_dummies(df, columns=low_card, drop_first=True, dtype=int)
    print(f"One-hot encoding applied")

# Label encode high-cardinality columns
label_encoders = {}
for col in high_card:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

if label_encoders:
    with open(ARTEFACTS + 'label_encoders.pkl', 'wb') as f:
        pickle.dump(label_encoders, f)
    print(f"Label encoding applied and saved")

print(f"\nDataset shape after encoding: {df.shape}")
mimic_dfs['merged'] = df

Low-cardinality (one-hot):  ['admission_type', 'insurance', 'marital_status', 'age_group']
High-cardinality (ordinal): ['admit_provider_id', 'admission_location', 'language', 'race']
One-hot encoding applied
Label encoding applied and saved

Dataset shape after encoding: (546028, 31)


## Stage 12 — Feature Selection (Variance → Correlation → Mutual Information)

In [ ]:
df = mimic_dfs['merged'].copy()

# Separate features from identifiers and outcome
id_cols_to_drop = [c for c in [SUBJECT_COL, HADM_COL, 'stay_id'] if c in df.columns]
feature_cols = [c for c in df.columns if c not in id_cols_to_drop + [OUTCOME_COL]]

X_all = df[feature_cols].values.astype(np.float32)
y_all = df[OUTCOME_COL].values.astype(np.int64)

print(f"Starting features: {len(feature_cols)}")

# Step 1 — Variance threshold
var_sel  = VarianceThreshold(threshold=0.01)
X_var    = var_sel.fit_transform(X_all)
feat_var = [feature_cols[i] for i,v in enumerate(var_sel.get_support()) if v]
print(f"After variance filter:    {len(feat_var):>4} features  "
      f"({len(feature_cols)-len(feat_var)} removed)")

# Step 2 — Correlation filter
CORR_THRESHOLD = 0.95
corr_matrix = pd.DataFrame(X_var, columns=feat_var).corr().abs()
upper       = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_corr   = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
feat_corr   = [c for c in feat_var if c not in drop_corr]
X_corr      = pd.DataFrame(X_var, columns=feat_var)[feat_corr].values
print(f"After correlation filter: {len(feat_corr):>4} features  "
      f"({len(drop_corr)} removed, threshold={CORR_THRESHOLD})")

# Step 3 — Mutual Information (top 100)
MAX_FEATURES = min(100, len(feat_corr))
print(f"Computing mutual information scores (top {MAX_FEATURES})...")
mi_scores    = mutual_info_classif(X_corr, y_all, random_state=RANDOM_SEED)
mi_df        = pd.DataFrame({'feature':feat_corr,'mi_score':mi_scores})
mi_df        = mi_df.sort_values('mi_score', ascending=False).reset_index(drop=True)
top_features = mi_df.head(MAX_FEATURES)['feature'].tolist()
X_selected   = pd.DataFrame(X_corr, columns=feat_corr)[top_features].values
print(f"After MI selection:       {len(top_features):>4} features (top {MAX_FEATURES})")

print("\nTop 15 most informative features:")
display(mi_df.head(15))

mi_df.to_csv(ARTEFACTS + 'feature_importance.csv', index=False)
with open(ARTEFACTS + 'selected_features.json', 'w') as f:
    json.dump(top_features, f, indent=2)
print("\nFeature selection artefacts saved to Drive")

Starting features: 27
After variance filter:      26 features  (1 removed)
After correlation filter:   26 features  (0 removed, threshold=0.95)
Computing mutual information scores (top 26)...
After MI selection:         26 features (top 26)

Top 15 most informative features:


,feature,mi_score
0,gender_female,0.0885
1,language,0.0757
2,insurance_Medicare,0.0739
3,marital_status_MARRIED,0.0676
4,race,0.0576
5,marital_status_SINGLE,0.0516
6,admission_location,0.0380
7,admission_type_EW EMER.,0.0380
8,insurance_Private,0.0373
9,age_group_41-60,0.0361



Feature selection artefacts saved to Drive


## Stage 13 — Class Imbalance Assessment

In [ ]:
class_counts = pd.Series(y_all).value_counts().sort_index()
ratio = class_counts[0] / class_counts.get(1, 1)

print("CLASS DISTRIBUTION")
print("="*40)
print(f"Survived (0): {class_counts[0]:>8,}  ({class_counts[0]/len(y_all)*100:.1f}%)")
print(f"Died     (1): {class_counts.get(1,0):>8,}  ({class_counts.get(1,0)/len(y_all)*100:.1f}%)")
print(f"Imbalance ratio: {ratio:.1f}:1")

# Compute class weights for FL training loss function
cw0 = len(y_all) / (2 * class_counts[0])
cw1 = len(y_all) / (2 * class_counts.get(1,1))
class_weights = {'0': round(cw0,4), '1': round(cw1,4)}

print(f"\nClass weights for weighted loss:")
print(f"  Class 0 weight: {cw0:.4f}")
print(f"  Class 1 weight: {cw1:.4f}")

with open(ARTEFACTS + 'class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)
print("\nClass weights saved to Drive (used in Notebook 02 FL simulation)")

CLASS DISTRIBUTION
Survived (0):  534,227  (97.8%)
Died     (1):   11,801  (2.2%)
Imbalance ratio: 45.3:1

Class weights for weighted loss:
  Class 0 weight: 0.5110
  Class 1 weight: 23.1348

Class weights saved to Drive (used in Notebook 02 FL simulation)


## Stage 14 — Normalisation, Train/Test Split, and Save

In [ ]:
# Train/test split BEFORE scaling — prevents data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_all,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_all       # preserve class ratio in both splits
)

# Fit StandardScaler on training set ONLY
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)       # transform only — never refit

# ── Validation checks ─────────────────────────────────────────────────────
print("VALIDATION CHECKS")
print("="*40)
checks = [
    ("No NaN in train features",   np.isnan(X_train_sc).sum() == 0),
    ("No NaN in test features",    np.isnan(X_test_sc).sum() == 0),
    ("No Inf in train features",   np.isinf(X_train_sc).sum() == 0),
    ("Outcome is binary",          set(np.unique(y_train)).issubset({0,1})),
    ("Class ratio consistent",     abs(y_train.mean()-y_test.mean()) < 0.02),
    ("Scaling mean ~0",            abs(X_train_sc.mean()) < 0.01),
    ("Scaling std ~1",             abs(X_train_sc.std()-1.0) < 0.05),
    ("Train > test",               len(X_train_sc) > len(X_test_sc)),
    ("Feature count <= 100",       X_train_sc.shape[1] <= 100),
    ("Sufficient samples (>500)",  len(X_train_sc) > 500),
]

all_passed = True
for name, result in checks:
    icon = "[OK]" if result else "[FAIL]"
    print(f"  {icon}  {name}")
    if not result: all_passed = False

print(f"\n{'All checks PASSED' if all_passed else 'WARNING: some checks FAILED — review above'}")

# ── Save to Google Drive ──────────────────────────────────────────────────
np.save(CLEAN_DIR + 'X_train.npy', X_train_sc)
np.save(CLEAN_DIR + 'X_test.npy',  X_test_sc)
np.save(CLEAN_DIR + 'y_train.npy', y_train)
np.save(CLEAN_DIR + 'y_test.npy',  y_test)

with open(ARTEFACTS + 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

metadata = {
    'input_dim':         int(X_train_sc.shape[1]),
    'n_train':           int(len(X_train_sc)),
    'n_test':            int(len(X_test_sc)),
    'outcome_col':       OUTCOME_COL,
    'positive_rate':     float(y_train.mean()),
    'selected_features': top_features,
    'class_weights':     class_weights,
    'zip_source':        ZIP_PATH,
    'data_subfolder':    DATA_SUBFOLDER,
}
with open(ARTEFACTS + 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSAVED TO GOOGLE DRIVE")
print(f"  Training set:  X_train.npy  {X_train_sc.shape}")
print(f"  Test set:      X_test.npy   {X_test_sc.shape}")
print(f"  Labels:        y_train.npy, y_test.npy")
print(f"  Scaler:        scaler.pkl")
print(f"  Metadata:      metadata.json")
print(f"  Location:      {CLEAN_DIR}")
print(f"\nPreprocessing pipeline COMPLETE")
print(f"Memory usage: {get_memory_mb():.1f} MB")
print(f"\nProceed to Notebook 02 — FL Simulation")

VALIDATION CHECKS
  [OK]  No NaN in train features
  [OK]  No NaN in test features
  [OK]  No Inf in train features
  [OK]  Outcome is binary
  [OK]  Class ratio consistent
  [OK]  Scaling mean ~0
  [OK]  Scaling std ~1
  [OK]  Train > test
  [OK]  Feature count <= 100
  [OK]  Sufficient samples (>500)

All checks PASSED

SAVED TO GOOGLE DRIVE
  Training set:  X_train.npy  (436822, 26)
  Test set:      X_test.npy   (109206, 26)
  Labels:        y_train.npy, y_test.npy
  Scaler:        scaler.pkl
  Metadata:      metadata.json
  Location:      /content/drive/MyDrive/FL_Dissertation/data/cleaned/

Preprocessing pipeline COMPLETE
Memory usage: 1454.5 MB

Proceed to Notebook 02 — FL Simulation
